### Within-Family Diagnostic — EC-NAS only

Same diagnostic as `03a_within_butter_e.ipynb`: Random Forest, log1p(target), standard random 80/20 split within EC-NAS rows only (not pooled). Also compares against EC-NAS's own reported surrogate-model metric, pulled directly from the paper (Bakhtiarifard et al., *EC-NAS: Energy Consumption Aware Tabular Benchmarks for Neural Architecture Search*, arXiv:2210.06015).

In [ ]:
# IMPORTS & LOAD

import sys

import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split

sys.path.insert(0, "../../")
from src.models import random_forest

df = pd.read_csv("../../data/processed/combined/combined_features.csv")
ecnas = df[df["family"] == "CNN"]

FEATURES = ["params", "depth", "flops", "epochs", "batch_size"]
TARGET = "target"
ecnas.shape

In [ ]:
# TRAIN & EVALUATE — RF, log1p(target), within-family split only

X = ecnas[FEATURES]
y = ecnas[TARGET]
y_log = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
_, _, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

model = random_forest.build_model()
model.fit(X_train, y_train_log)
preds_log = model.predict(X_test)
preds_raw = np.expm1(preds_log)

mape_log = mean_absolute_percentage_error(y_test_log, preds_log)
r2_log = r2_score(y_test_log, preds_log)
mape_raw = mean_absolute_percentage_error(y_test, preds_raw)
r2_raw = r2_score(y_test, preds_raw)
tau, tau_p = kendalltau(y_test, preds_raw)

print(f"n = {len(ecnas)}")
print(f"MAPE (log scale):  {mape_log:.4f}")
print(f"R2   (log scale):  {r2_log:.4f}")
print(f"MAPE (raw, expm1): {mape_raw:.4f}")
print(f"R2   (raw, expm1): {r2_raw:.4f}")
print(f"Kendall-Tau:       {tau:.4f}  (p={tau_p:.2e})")

**Result** (n=2,805): MAPE (log) = 0.005, R² (log) = 0.955, MAPE (raw) = 0.056, R² (raw) = 0.958, **Kendall-Tau = 0.861** (p≈4×10⁻²⁰³).

This family alone is predicted well — the same 5 features that struggled on BUTTER-E work fine here (R² ≈0.96, MAPE ≈5.6%). The pooled bake-off's weak numbers are not a general failure of the feature set or the pooling approach; they're driven almost entirely by BUTTER-E (see `03a_within_butter_e.ipynb`).

**Comparison against EC-NAS's own reported metric — not quite apples-to-apples, worth stating plainly.** Their paper (Fig. 2, §2.3/Appendix C) reports "Kendall-Tau R² = 0.9030" for their own surrogate MLP (`36→128→64→32→1`, GELU, L1 loss, Adam, lr=5×10⁻³, 200 epochs). But that model was trained and evaluated on the **7V space's energy labels — which are themselves surrogate-predicted/linearly-extrapolated values, not real measurements** (the paper's own appendix: "the 7V space[, where] the resource costs for the remaining epochs [are] extrapolated through linear scaling," 4,310 sampled architectures, 3020/430/860 train/val/test split). It's a model learning to reproduce another model's synthetic output — a smoother, easier target than real measured energy.

Our 0.861 Kendall-Tau, by contrast, is against **real, hardware-measured** energy (Carbontracker readings on actual training runs, the 4V/5V direct-measurement subset — see `02b_ec_nas_features.ipynb`). Given it's a harder, noisier target, 0.861 against their 0.903 is a genuinely favorable result, not a shortfall — just not a claim that can be stated as "we matched their number" without this caveat attached.